In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Let's start with necessary imports
import os
import numpy as np
import pandas as pd
from shutil import copyfile
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt

## Define Settings

In [3]:
data_dir = "../../data/images/"
meta_data_path = "../../data/butterfly_anomaly_train.csv"
plot_dir = "../../plots/"
model_dir = "../../data/models/"

In [4]:
for dir_name in [plot_dir, model_dir]:
    if not os.path.exists(dir_name):
        os.makedirs(dir_name)

In [5]:
from hdr_hybrid_butterflies import config

## Define Data Loader

In [ ]:
from hdr_hybrid_butterflies.data_handler import DataHandler, ImageProcessor

data_handler = DataHandler(
    meta_data_path=meta_data_path,
    data_dir=data_dir,
)

In [ ]:
data_handler.df_meta.iloc[10]["hybrid_stat"] == "nonhybrid"

In [ ]:
data_handler.df_meta.iloc[10]

In [ ]:
data_handler.load_data(1804)[0]

### Copy Files of Subspecies to sub-directories

In [ ]:
np.unique(data_handler.df_meta["subspecies"], return_counts=True)

In [11]:
if False:
    for idx, row in tqdm(
        data_handler.df_meta.iterrows(), total=data_handler.n_samples
    ):
        if row["hybrid_stat"] == "non-hybrid":
            input_path = os.path.join(
                data_dir, row["hybrid_stat"], row["filename"]
            )
            output_path = os.path.join(
                data_dir,
                row["hybrid_stat"],
                f"{int(row['subspecies']):02d}",
                row["filename"],
            )
            output_dir = os.path.dirname(output_path)
            if not os.path.exists(output_dir):
                os.makedirs(output_dir)
            copyfile(input_path, output_path)

## Create overview of all Subspecies

In [ ]:
n_subspecies = len(np.unique(data_handler.df_meta["subspecies"]))
n_subspecies

In [ ]:
data_handler.df_meta[
    data_handler.df_meta["subspecies"].isnull()
].parent_subspecies_2.unique()

In [14]:
if False:
    for seed in range(10):
        rng = np.random.default_rng(seed)

        sub_species = np.unique(data_handler.df_meta["subspecies"])

        fig, axes = plt.subplots(3, 5, figsize=(30, 15))
        axes_flat = axes.flatten()

        for idx, sub in enumerate(sub_species):
            if np.isnan(sub):
                indices = data_handler.df_meta[
                    data_handler.df_meta["subspecies"].isnull()
                ].index
            else:
                indices = data_handler.df_meta[
                    data_handler.df_meta["subspecies"] == sub
                ].index

            chosen_idx = rng.choice(indices)

            img, row = data_handler.load_data(chosen_idx)
            axes_flat[idx].imshow(img)
            axes_flat[idx].set_title(
                f"Subspecies: {row['subspecies']} | Idx: {chosen_idx} | Occurrences: {len(indices)}"
            )
            axes_flat[idx].axis("off")

        plt.tight_layout()
        fig.savefig(
            os.path.join(plot_dir, f"subspecies_examples_{seed:04d}.png")
        )

## Test Grounded Segment Anything (Grounded DINO + SAM)

In [ ]:
image, row = data_handler.load_data(874)
image

In [ ]:
from PIL import ImageOps

# image, row = data_handler.load_by_name("CAM008547.jpg")
# image, row = data_handler.load_by_name("CAM011441")
image, row = data_handler.load_by_name("CAM036588")
image = ImageOps.exif_transpose(image)
image = ImageOps.exif_transpose(image)
image = ImageOps.exif_transpose(image)
image = ImageOps.exif_transpose(image)
image = ImageOps.exif_transpose(image)
image

In [ ]:
from hdr_hybrid_butterflies.dino_sam.dino_sam import grounded_segmentation

In [18]:
# image = segment_data_handler.load_data(1834)[0]
# image

In [ ]:
1237
# labels = ["upper left wing.", "lower left wing.", "upper right wing.", "lower right wing."]
labels = [
    "upper left butterfly wing.",
    "lower left butterfly wing.",
    "upper right butterfly wing.",
    "lower right butterfly wing.",
]
# labels = ["upper wing.", "lower wing."]
# labels = ["left wing.", "right wing."]
labels = ["wings."]
# labels = ["upper wing.", "lower wing."]
# labels = ["larger wing.", "smaller wing."]
# labels = ["butterfly wing."]
# labels = ["Orange."]
threshold = 0.2

detector_id = "IDEA-Research/grounding-dino-tiny"
segmenter_id = "facebook/sam-vit-base"


image_array, detections = grounded_segmentation(
    image=image,
    labels=labels,
    threshold=threshold,
    polygon_refinement=True,
    detector_id=detector_id,
    segmenter_id=segmenter_id,
)

In [ ]:
600 - 2 * 60 + 5

In [21]:
from hdr_hybrid_butterflies.dino_sam.dino_sam import BoundingBox
from transformers import AutoModelForMaskGeneration, AutoProcessor, pipeline

segmentator = AutoModelForMaskGeneration.from_pretrained(segmenter_id).to(
    "cpu"
)
processor = AutoProcessor.from_pretrained(segmenter_id)

In [ ]:
boxes = [
    [
        [
            0,
            0,
        ]
        + list(image_array.shape[:2])
    ]
]
boxes = [[[0, 0, 500, 500]]]
inputs = processor(images=image, input_boxes=boxes, return_tensors="pt").to(
    "cpu"
)
inputs

In [23]:
outputs = segmentator(**inputs)

In [24]:
from hdr_hybrid_butterflies.dino_sam import plotting

In [ ]:
def get_most_confident_detection(detections, label):
    detections_mask = [d for d in detections if d.label == label]
    return sorted(detections_mask, key=lambda x: x.score)[-1]


detections_confident = [
    get_most_confident_detection(detections, label) for label in labels
]
detections_confident

In [ ]:
fig, ax = plotting.plot_detections(image_array, detections)
fig.savefig(os.path.join(plot_dir, "detections.png"))

In [ ]:
fig, ax = plotting.plot_detections(image_array, detections_confident)

## Create training data for segment classifier 

Classified created image segments into one of the following classes: 
    upper / lower / noise
Further steps will only be run on segments that pass the classifier

In [28]:
segment_training_dir = os.path.join(data_dir, "segment_classier_training")

if not os.path.exists(segment_training_dir):
    os.makedirs(segment_training_dir)

In [ ]:
from PIL import Image
from glob import glob

data_processor = ImageProcessor()

for idx, row in tqdm(
    data_handler.df_meta.iterrows(), total=data_handler.n_samples
):
    file_glob = f"{row['filename'][:-4]}_*.jpg"
    if len(glob(os.path.join(segment_training_dir, "all", file_glob))) > 0:
        print(f"Skipping {row['filename']}")
        continue

    image, row = data_handler.load_data(idx)
    _, _, segments, scores = data_processor._raw_segments(image)

    for idx_j, segment in enumerate(segments):
        score = scores[idx_j]
        filename = f"{row['filename'][:-4]}_{idx_j:04d}_s{score:0.4f}.jpg"
        PIL_image = Image.fromarray(segment)
        PIL_image.save(os.path.join(segment_training_dir, "all", filename))

Re-create segments for individual input files

In [30]:
if False:
    input_file_glob = "../../data/images/non-hybrid/11/*.jpg"
    segment_individual_dir = os.path.join(segment_training_dir, "individual")

    if not os.path.exists(segment_individual_dir):
        os.makedirs(segment_individual_dir)

    input_files = sorted(glob(input_file_glob))
    for input_file in tqdm(input_files, total=len(input_files)):
        image = Image.open(input_file)
        image = ImageOps.exif_transpose(image)

        camid = os.path.basename(input_file)[:-4]

        _, _, segments, scores = data_processor._raw_segments(image)

        for idx_j, segment in enumerate(segments):
            score = scores[idx_j]
            filename = f"{camid}_{idx_j:04d}_s{score:0.4f}.jpg"
            PIL_image = Image.fromarray(segment)
            PIL_image.save(os.path.join(segment_individual_dir, filename))

Separate segments out into upper, lower, and noise directories

In [31]:
upper_wing_dir = os.path.join(segment_training_dir, "upper_wing")
lower_wing_dir = os.path.join(segment_training_dir, "lower_wing")
noise_dir = os.path.join(segment_training_dir, "noise")

for output_dir in [upper_wing_dir, lower_wing_dir, noise_dir]:
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

In [32]:
import imagesize

if False:
    segment_files = sorted(
        glob(os.path.join(segment_training_dir, "all", "*.jpg"))
    )

    for idx, filename in enumerate(segment_files):
        width, height = imagesize.get(filename)
        ratio = width / height
        score = float(filename.split("_")[-1][1:-4])
        if score > 0.5:
            if ratio > 1.4:
                copyfile(
                    filename,
                    os.path.join(upper_wing_dir, os.path.basename(filename)),
                )
            else:
                copyfile(
                    filename,
                    os.path.join(lower_wing_dir, os.path.basename(filename)),
                )
        elif score > 0.1:
            copyfile(
                filename, os.path.join(noise_dir, os.path.basename(filename))
            )

        print(f"{idx:04d} | {ratio:0.3f} | {width}x{height} | {score:0.4f}")

    len(segment_files)

In [ ]:
from hdr_hybrid_butterflies.data_handler import (
    SegmentDataHandler,
    ImageProcessor,
)

image_processor = ImageProcessor()

segment_data_handler = SegmentDataHandler(
    meta_data_path=meta_data_path,
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    data_dir_noise=os.path.join(
        segment_training_dir, "manual", "noise_manual"
    ),
    image_processor=image_processor,
)
segment, segment_info = segment_data_handler.load_data(10)
segment

In [34]:
segment_generator_test = segment_data_handler.get_generator(
    batch_size=16,
    queue_size=160,  # 1600,
    n_jobs=1,
    mask_only=True,
    training=False,
)

In [ ]:
segments, segment_labels = next(segment_generator_test)
fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes_flat = axes.flatten()
for idx, ax in enumerate(axes_flat):
    ax.imshow(segments[idx])
    ax.set_title(f"Label: {segment_labels[idx]}")
    ax.axis("off")

In [ ]:
%timeit segment_data_handler()

## Train Segment Classifier

In [38]:
segment_generator_train = segment_data_handler.get_generator(
    batch_size=16,
    queue_size=10000,
    n_jobs=12,
    mask_only=True,
    training=True,
)

In [ ]:
image_processor.output_dim

In [ ]:
import tensorflow as tf
from hdr_hybrid_butterflies.model import CNNClasifier

model = CNNClasifier(
    image_size=image_processor.output_dim,
    num_classes=3,
)
print(model.call(segments).shape)
model.summary()

In [38]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)

In [39]:
segment_model_dir = os.path.join(model_dir, "segment_model")

In [ ]:
model.load_weights(os.path.join(segment_model_dir, "model.weights.h5"))

In [36]:
# Create the ModelCheckpoint callback
checkpoint_path = os.path.join(segment_model_dir, "model.weights.h5")

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    monitor="loss",  # 'val_loss'
    mode="min",
    save_best_only=False,
    save_freq="epoch",  # Save every epoch
    verbose=1,
)

In [ ]:
steps_per_epoch = 100
epochs = 100


model.fit(
    segment_generator_train,
    validation_data=segment_generator_test,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_steps=5,
    callbacks=[model_checkpoint_callback],
)

#### Evaluate Segmentation Model

In [ ]:
segments, segment_labels = next(segment_generator_test)

# get prediction
logits = model(segments)
probs = model.logits2probs(logits)

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes_flat = axes.flatten()
for idx, ax in enumerate(axes_flat):
    ax.imshow(segments[idx])
    ax.set_title(
        f"Label: {segment_labels[idx]} | Pred: {np.argmax(probs[idx])} ["
        f"{probs[idx][0]:0.2f}, {probs[idx][1]:0.2f}, {probs[idx][2]:0.2f}]"
    )
    ax.axis("off")

In [46]:
%matplotlib inline

In [ ]:
from sklearn.metrics import classification_report

n_steps = 100

incorrect_segments = []
incorrect_labels = []

predictions = []
labels = []
for idx in tqdm(range(n_steps), total=n_steps):
    segments, segment_labels = next(segment_generator_test)
    prediction = np.argmax(model.logits2probs(model(segments)), axis=-1)

    mask = prediction != segment_labels
    if np.any(mask):
        incorrect_segments.append(segments[mask])
        incorrect_labels.append(segment_labels[mask])

    predictions.append(prediction)
    labels.append(segment_labels)

predictions = np.concatenate(predictions)
labels = np.concatenate(labels)

print(classification_report(labels, predictions))

incorrect_labels = np.concatenate(incorrect_labels)
incorrect_segments = np.concatenate(incorrect_segments)

In [ ]:
predictions = model.logits2probs(model(incorrect_segments))
for label, segment, prediction in zip(
    incorrect_labels, incorrect_segments, predictions
):
    plt.imshow(segment)
    plt.title(
        f"Label: {label} | Pred: {np.argmax(prediction)} "
        f"[{prediction[0]:0.2f}, {prediction[1]:0.2f}, {prediction[2]:0.2f}]"
    )
    plt.axis("off")
    plt.show()

In [ ]:
%timeit model(segments)

In [ ]:
del segment_data_handler
del segment_generator_train
del segment_generator_test

# garbage collect
import gc

gc.collect()

#### Test image processing pipeline

In [ ]:
image_processor_test = ImageProcessor(segment_classifier=model)

In [ ]:
image = data_handler.load_data(2000)[0]
# image = data_handler.load_by_name("CAM000446")[0]
image = data_handler.load_by_name("CAM016768")[0]
image

In [ ]:
lower_segments, upper_segments = image_processor_test(image)

In [ ]:
for segment in lower_segments:
    plt.imshow(segment)
    plt.axis("off")
    plt.show()

In [ ]:
for segment in upper_segments:
    plt.imshow(segment)
    plt.axis("off")
    plt.show()

In [ ]:
hybrid_classifier = CNNClasifier(
    image_size=image_processor_test.output_dim,
    num_classes=2,
    verbose=False,
)
hybrid_classifier.load_weights(
    os.path.join(model_dir, "signal_hybrid_model", "model.weights.h5")
)

probabilities = hybrid_classifier.probabilities(
    np.stack(upper_segments)
).numpy()
probabilities

In [ ]:
np.mean(probabilities, axis=0)[1]

## Create training data for Segmentation

Classify pixels in input image to True/False. True is part of wing. Potentially split in upper and lower wing pixels.

In [40]:
segmentation_training_dir = os.path.join(data_dir, "segmentation_training")

if not os.path.exists(segmentation_training_dir):
    os.makedirs(segmentation_training_dir)

In [41]:
# create labels for each input image in the form of masks for upper/lower/all.
# Use segment classifier to create these labels
# --> e.g. we want to try and reproduce what we have with the time consuming route

In [ ]:
from PIL import Image
from glob import glob
import pickle

data_processor = ImageProcessor(segment_classifier=model)

for idx, row in tqdm(
    data_handler.df_meta.iterrows(), total=data_handler.n_samples
):
    if not os.path.exists(
        os.path.join(data_dir, row["hybrid_stat"], row["filename"])
    ):
        print(f"File not found: {row['filename'], row['hybrid_stat']}")
        continue

    file_glob = f"{row['filename'][:-4]}_*.*"
    if len(glob(os.path.join(segmentation_training_dir, file_glob))) == 3:
        # print(f"Skipping {row['filename']}")
        continue

    image, row = data_handler.load_data(idx)
    (
        image_array,
        mask_array,
        boxes_upper,
        boxes_lower,
    ) = data_processor._raw_segmentation_labels(image)
    for suffix, img_arr, quality in [
        ("mask", mask_array, 100),
        ("image", image_array, 75),
    ]:
        filename = f"{row['filename'][:-4]}_{suffix}.jpg"
        PIL_image = Image.fromarray(img_arr)
        PIL_image.save(
            os.path.join(segmentation_training_dir, filename), quality=quality
        )

    with open(
        os.path.join(
            segmentation_training_dir, f"{row['filename'][:-4]}_box.pkl"
        ),
        "wb",
    ) as f:
        pickle.dump((boxes_upper, boxes_lower), f)

## Train Image Segmenter

In [ ]:
from hdr_hybrid_butterflies.data_handler import SegmentationDataHandler

data_processor = ImageProcessor(
    segment_classifier=model,
    p_erase=0.0,
    padding_size=0,
    output_dim=config.CNN_SEGMENTER_IMAGE_SIZE,
)
data_handler_segmentation = SegmentationDataHandler(
    meta_data_path=meta_data_path,
    data_dir=segmentation_training_dir,
    image_processor=data_processor,
    reduction_factor=4,
)
img_aug, mask_aug, row = data_handler_segmentation(grayscale=True)
img_aug.shape, mask_aug.shape

In [ ]:
img_aug, mask_aug, row = data_handler_segmentation()
plt.imshow(img_aug)
plt.show()
# Image.fromarray(img_aug)

In [ ]:
# Image.fromarray(mask_aug)
plt.imshow(mask_aug)

#### Verify contours

In [ ]:
import cv2

img_aug, mask_aug, row = data_handler_segmentation()

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.imshow(img_aug)

# get contours of mask
contours, _ = cv2.findContours(
    mask_aug, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)

# draw contours
for contour in contours:
    ax.plot(contour[:, 0, 0], contour[:, 0, 1], color="red", linewidth=2)

In [ ]:
%timeit data_handler_segmentation(grayscale=False)

In [ ]:
img_aug.shape

In [ ]:
from hdr_hybrid_butterflies.model import CNNSegmenter

cnn_segmenter = CNNSegmenter(num_classes=3, image_size=img_aug.shape[:2])
cnn_segmenter.summary()
cnn_segmenter.probabilities(img_aug[None, ...]).shape

In [56]:
img_aug, mask_aug, row = data_handler_segmentation()

In [ ]:
%timeit cnn_segmenter.probabilities(img_aug[None, ...])

In [ ]:
cnn_segmenter.load_weights(
    os.path.join(model_dir, "segmentation_model", "model.weights.h5")
)

In [204]:
%matplotlib inline

In [ ]:
img_aug, mask_aug, row = data_handler_segmentation()
plt.imshow(img_aug)
plt.show()

In [ ]:
probs = cnn_segmenter.probabilities(img_aug[None, ...])
plt.imshow(probs[0])
plt.show()

In [ ]:
image = data_handler.load_data(2001)[0]
image

In [ ]:
cnn_segmenter_processor = ImageProcessor(
    segment_classifier=model,
    p_erase=0.0,
    padding_size=0,
    output_dim=config.CNN_SEGMENTER_IMAGE_SIZE,
)

full_processor = ImageProcessor(
    segment_classifier=model,
    cnn_segmenter=cnn_segmenter,
    cnn_segmenter_processor=cnn_segmenter_processor,
)

lower_segments, upper_segments = full_processor(image, via_cnn=True)

In [ ]:
%timeit lower_segments, upper_segments = full_processor(image, via_cnn=True)

In [ ]:
image

In [ ]:
for segment in lower_segments:
    print(segment.shape)
    plt.imshow(segment)
    plt.axis("off")
    plt.show()
for segment in upper_segments:
    plt.imshow(segment)
    plt.axis("off")
    plt.show()

In [ ]:
for segment in lower_segments:
    print(segment.shape)
    plt.imshow(segment)
    plt.axis("off")
    plt.show()

for segment in upper_segments:
    plt.imshow(segment)
    plt.axis("off")
    plt.show()

## Create Training Data for Features

In [ ]:
image_array_segment = segment_data_handler(apply_augmentations=False)[0]
segment_im = Image.fromarray(image_array_segment)
segment_im

In [ ]:
from hdr_hybrid_butterflies.data_utils import (
    get_dominant_colors,
    extract_features,
)

results = extract_features(image_array_segment)
for result in results:
    plt.imshow(result)
    plt.show()

In [689]:
# new_img = np.array(image_array_segment)
# new_img[np.any(result1 > 10, axis=-1)] = result1[np.any(result1 > 10, axis=-1)]
# plt.imshow(new_img)

In [ ]:
%timeit extract_features(image_array_segment)

### Create training data for features

In [321]:
feature_dir = os.path.join(data_dir, "features_training")

if not os.path.exists(feature_dir):
    os.makedirs(feature_dir)

for i in range(14):
    for label in ["upper", "lower", "noise"]:
        feature_dir_sub = os.path.join(feature_dir, f"{i:02d}", label)
        if not os.path.exists(feature_dir_sub):
            os.makedirs(feature_dir_sub)

In [ ]:
row

In [ ]:
for idx in tqdm(range(segment_data_handler.n_samples)):
    image_array_segment, row = segment_data_handler.load_data(idx)

    image_array_segment = segment_data_handler.image_processor.augment_image(
        image_array_segment,
        mask_only=False,
        grayscale=False,
        apply_augmentations=False,
    )[0]
    features = extract_features(image_array_segment)
    feature_dir_i = os.path.join(
        feature_dir, f"{int(row['subspecies']):02d}", row["label"]
    )

    name = row["filename"].split("_s")[0]

    for idx, feature in enumerate(features):
        filename = f"{name}_f{idx:02d}.jpg"
        PIL_image = Image.fromarray(feature)
        PIL_image.save(os.path.join(feature_dir_i, filename), quality=100)

In [ ]:
features = extract_features(image_array_segment)

for idx, feature in enumerate(features):
    filename = f"{row['filename']}_f{idx:02d}.jpg"
    PIL_image = Image.fromarray(feature)
    PIL_image.save(os.path.join(feature_dir_i, filename), quality=100)

In [ ]:
1237
# labels = ["upper left wing.", "lower left wing.", "upper right wing.", "lower right wing."]
labels = [
    "upper left butterfly wing.",
    "lower left butterfly wing.",
    "upper right butterfly wing.",
    "lower right butterfly wing.",
]
# labels = ["upper wing.", "lower wing."]
# labels = ["left wing.", "right wing."]
labels = ["orange pattern."]
# labels = ["upper wing.", "lower wing."]
# labels = ["larger wing.", "smaller wing."]
# labels = ["butterfly wing."]
# labels = ["Orange."]
threshold = 0.2

detector_id = "IDEA-Research/grounding-dino-tiny"
segmenter_id = "facebook/sam-vit-base"


image_array_segment, detections_segment = grounded_segmentation(
    image=segment_im,
    labels=labels,
    threshold=threshold,
    polygon_refinement=True,
    detector_id=detector_id,
    segmenter_id=segmenter_id,
)

In [ ]:
target_color = np.array([233, 124, 37])  # Example: Orange
# target_color = np.array([238, 246, 253])  # Example: white
tolerance = np.array([50, 50, 50])

lower_bound = target_color - tolerance
upper_bound = target_color + tolerance

# image = cv2.cvtColor(image_array_segment, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(image_array_segment, cv2.COLOR_RGB2HSV)
mask = cv2.inRange(image_array_segment, lower_bound, upper_bound)
result = cv2.bitwise_and(image, image, mask=mask)
plt.imshow(result)

In [ ]:
fig, ax = plotting.plot_detections(
    image_array_segment, detections_segment, width=2
)

## Test Segment Stiching

In [ ]:
wing1 = segment_data_handler(apply_augmentations=False)[0]
plt.imshow(wing1)

In [ ]:
wing2 = segment_data_handler(apply_augmentations=False)[0]
plt.imshow(wing2)

In [ ]:
from hdr_hybrid_butterflies.data_utils import (
    get_dominant_colors,
    extract_features,
    rotate_image,
)


wing1_rotated, size = rotate_image(wing1)
plt.imshow(wing1_rotated)
size
# result = get_best_alignment(wing1, wing2)
# plt.imshow(result)

# compute_overlap(wing1, wing2), compute_overlap(wing1, result)

In [ ]:
from hdr_hybrid_butterflies.data_utils import (
    overlay_images_max,
    overlay_images,
    create_hybrid_image,
)

hybrid, hybrid_label = create_hybrid_image(wing1, wing2)
plt.imshow(hybrid)
plt.title(hybrid_label)


# overlay = overlay_images(wing1, wing2)
# overlay = overlay_images_max(wing1, wing2)


# plt.imshow(overlay)

## Test FeatureWingDataGenerators

In [ ]:
from hdr_hybrid_butterflies.data_handler import (
    UpperFeatureWingDataHandler,
    LowerFeatureWingDataHandler,
)

p_erase = 0.1
feature_upper_data_handler = UpperFeatureWingDataHandler(
    meta_data_path=meta_data_path,
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    data_dir_features=os.path.join(data_dir, "features_training", "manual"),
    image_processor=ImageProcessor(p_erase=p_erase),
)

feature_lower_data_handler = LowerFeatureWingDataHandler(
    meta_data_path=meta_data_path,
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    data_dir_features=os.path.join(data_dir, "features_training", "manual"),
    image_processor=ImageProcessor(p_erase=p_erase),
)

In [ ]:
for feature_num in range(feature_upper_data_handler.num_features):
    print(
        feature_num, len(feature_upper_data_handler.feature_files[feature_num])
    )
print()

for feature_num in range(feature_lower_data_handler.num_features):
    print(
        feature_num, len(feature_lower_data_handler.feature_files[feature_num])
    )

In [ ]:
feature_upper_data_handler.load_feature(0, 1)

In [ ]:
feature_lower_data_handler.load_feature(0, 150)

In [ ]:
img, row = feature_upper_data_handler()
plt.imshow(img)
row

In [ ]:
img, row = feature_lower_data_handler()
plt.imshow(img)
row

## Test HybridStitcherWingSegmentDataHandler

In [ ]:
data_handler.df_meta["hybrid_stat"] == "non-hybrid"

In [ ]:
from hdr_hybrid_butterflies.data_handler import (
    HybridStitcherWingSegmentDataHandler,
)

hybrid_stitcher_data_handler = HybridStitcherWingSegmentDataHandler(
    meta_data_path=meta_data_path,
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    image_processor=ImageProcessor(),
)

In [ ]:
img_arrays, row = hybrid_stitcher_data_handler(
    apply_augmentations=False,
    grayscale=False,
)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for idx, ax in enumerate(axes.flatten()):
    ax.imshow(img_arrays[idx])
    ax.axis("off")
fig.suptitle(f"Hybrid Stitcher | {row['filename']} | {row['is_hybrid']}")
fig.tight_layout()
row

## Train Lower Wing Classifier

In [30]:
feature_number = 3

In [31]:
from hdr_hybrid_butterflies.data_handler import LowerWingDataHandler

lower_wing_data_handler = LowerWingDataHandler(
    meta_data_path=meta_data_path,
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    image_processor=image_processor,
)

In [ ]:
lower_wing_data_handler.n_samples

In [33]:
lower_wing_generator_train = lower_wing_data_handler.get_generator(
    batch_size=16,
    queue_size=10000,
    n_jobs=12,
    mask_only=False,
    training=True,
    labels_func_name=f"labels_feature_{feature_number:02d}",
)

lower_wing_generator_test = lower_wing_data_handler.get_generator(
    batch_size=16,
    queue_size=500,
    n_jobs=1,
    mask_only=False,
    training=False,
    labels_func_name=f"labels_feature_{feature_number:02d}",
)

In [ ]:
import tensorflow as tf
from hdr_hybrid_butterflies.model import CNNClasifier

model_lower = CNNClasifier(
    image_size=image_processor.output_dim,
    num_classes=2,
    name=f"lower_wing_feature_{feature_number:02d}",
)
print(model_lower.call(segments).shape)
model_lower.summary()

In [35]:
model_lower.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)

In [36]:
lower_model_dir = os.path.join(model_dir, f"lower_model_{feature_number:02d}")

In [ ]:
model_lower.load_weights(os.path.join(lower_model_dir, "model.weights.h5"))

In [38]:
# Create the ModelCheckpoint callback
checkpoint_path_lower = os.path.join(lower_model_dir, "model.weights.h5")

lower_model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path_lower,
    save_weights_only=True,
    monitor="loss",  # 'val_loss'
    mode="min",
    save_best_only=False,
    save_freq="epoch",  # Save every epoch
    verbose=1,
)

In [ ]:
steps_per_epoch = 100
epochs = 100


model_lower.fit(
    lower_wing_generator_train,
    validation_data=lower_wing_generator_test,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_steps=5,
    callbacks=[lower_model_checkpoint_callback],
)

#### Evaluate Lower Wing Model

In [40]:
%matplotlib inline

In [ ]:
segments, segment_labels = next(lower_wing_generator_test)

# get prediction
logits = model_lower(segments)
probs = model_lower.logits2probs(logits)

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes_flat = axes.flatten()
for idx, ax in enumerate(axes_flat):
    ax.imshow(segments[idx])
    ax.set_title(
        f"Label: {segment_labels[idx]} | Pred: {np.argmax(probs[idx])} ["
        f"{probs[idx][0]:0.2f}, {probs[idx][1]:0.2f}]"
    )

    # red frame around axes if prediction is wrong
    frame_color = (
        "green" if np.argmax(probs[idx]) == segment_labels[idx] else "red"
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(frame_color)
        spine.set_linewidth(7)

    # turn off everything except the spine (i.e. frame)
    ax.tick_params(
        axis="both",
        which="both",
        left=False,
        right=False,
        bottom=False,
        top=False,
        labelleft=False,
        labelbottom=False,
    )

In [ ]:
from sklearn.metrics import classification_report

n_steps = 100

incorrect_segments = []
incorrect_labels = []

predictions = []
labels = []
for idx in tqdm(range(n_steps), total=n_steps):
    segments, segment_labels = next(lower_wing_generator_test)
    prediction = np.argmax(
        model_lower.logits2probs(model_lower(segments)), axis=-1
    )

    mask = prediction != segment_labels
    if np.any(mask):
        incorrect_segments.append(segments[mask])
        incorrect_labels.append(segment_labels[mask])

    predictions.append(prediction)
    labels.append(segment_labels)

predictions = np.concatenate(predictions)
labels = np.concatenate(labels)

print(classification_report(labels, predictions))

incorrect_labels = np.concatenate(incorrect_labels)
incorrect_segments = np.concatenate(incorrect_segments)

In [ ]:
predictions = model_lower.logits2probs(model_lower(incorrect_segments))
for label, segment, prediction in zip(
    incorrect_labels, incorrect_segments, predictions
):
    plt.imshow(segment)
    plt.title(
        f"Label: {label} | Pred: {np.argmax(prediction)} "
        f"[{prediction[0]:0.2f}, {prediction[1]:0.2f}]"
    )
    plt.axis("off")
    plt.show()

## Train Upper Wing Classifier

In [41]:
feature_number = 10

In [42]:
from hdr_hybrid_butterflies.data_handler import UpperWingDataHandler

upper_wing_data_handler = UpperWingDataHandler(
    meta_data_path=meta_data_path,
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    image_processor=image_processor,
    # test_split=0.05,
)

In [ ]:
upper_wing_data_handler.n_samples

In [ ]:
result = np.unique(
    upper_wing_data_handler.df_meta["subspecies"].iloc[
        : upper_wing_data_handler.n_samples_train
    ],
    return_counts=True,
)
for sub, count in zip(*result):
    print(f"Subspecies: {sub} | Count: {count}")

In [ ]:
upper_wing_generator_train = upper_wing_data_handler.get_generator(
    batch_size=16,
    queue_size=10000,
    n_jobs=12,
    mask_only=False,
    training=True,
    balanced_loading=True,
    labels_func_name=f"labels_feature_{feature_number:02d}",
)

upper_wing_generator_test = upper_wing_data_handler.get_generator(
    batch_size=16,
    queue_size=1600,
    n_jobs=1,
    mask_only=False,
    training=False,
    labels_func_name=f"labels_feature_{feature_number:02d}",
)

In [ ]:
import tensorflow as tf
from hdr_hybrid_butterflies.model import CNNClasifier

model_upper = CNNClasifier(
    image_size=image_processor.output_dim,
    num_classes=2,
    name=f"upper_wing_feature_{feature_number:02d}",
)
print(model_upper.call(segments).shape)
model_upper.summary()

In [47]:
model_upper.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)

In [48]:
upper_model_dir = os.path.join(model_dir, f"upper_model_{feature_number:02d}")

In [ ]:
model_upper.load_weights(os.path.join(upper_model_dir, "model.weights.h5"))

In [50]:
# Create the ModelCheckpoint callback
checkpoint_path_upper = os.path.join(upper_model_dir, "model.weights.h5")

upper_model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path_upper,
    save_weights_only=True,
    monitor="loss",  # 'val_loss'
    mode="min",
    save_best_only=False,
    save_freq="epoch",  # Save every epoch
    verbose=1,
)

In [ ]:
steps_per_epoch = 100
epochs = 100


model_upper.fit(
    upper_wing_generator_train,
    validation_data=upper_wing_generator_test,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_steps=5,
    callbacks=[upper_model_checkpoint_callback],
)

#### Evaluate Upper Wing Model

In [45]:
%matplotlib inline

In [ ]:
segments, segment_labels = next(upper_wing_generator_test)

# get prediction
logits = model_upper(segments)
probs = model_upper.logits2probs(logits)

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes_flat = axes.flatten()
for idx, ax in enumerate(axes_flat):
    ax.imshow(segments[idx])
    ax.set_title(
        f"Label: {segment_labels[idx]} | Pred: {np.argmax(probs[idx])} ["
        f"{probs[idx][0]:0.2f}, {probs[idx][1]:0.2f}]"
    )

    # red frame around axes if prediction is wrong
    frame_color = (
        "green" if np.argmax(probs[idx]) == segment_labels[idx] else "red"
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(frame_color)
        spine.set_linewidth(7)

    # turn off everything except the spine (i.e. frame)
    ax.tick_params(
        axis="both",
        which="both",
        left=False,
        right=False,
        bottom=False,
        top=False,
        labelleft=False,
        labelbottom=False,
    )

In [ ]:
from sklearn.metrics import classification_report

n_steps = 100

incorrect_segments = []
incorrect_labels = []

predictions = []
labels = []
for idx in tqdm(range(n_steps), total=n_steps):
    segments, segment_labels = next(upper_wing_generator_test)
    prediction = np.argmax(
        model_upper.logits2probs(model_upper(segments)), axis=-1
    )

    mask = prediction != segment_labels
    if np.any(mask):
        incorrect_segments.append(segments[mask])
        incorrect_labels.append(segment_labels[mask])

    predictions.append(prediction)
    labels.append(segment_labels)

predictions = np.concatenate(predictions)
labels = np.concatenate(labels)

print(classification_report(labels, predictions))

incorrect_labels = np.concatenate(incorrect_labels)
incorrect_segments = np.concatenate(incorrect_segments)

In [ ]:
predictions = model_upper.logits2probs(model_upper(incorrect_segments))
for label, segment, prediction in zip(
    incorrect_labels, incorrect_segments, predictions
):
    plt.imshow(segment)
    plt.title(
        f"Label: {label} | Pred: {np.argmax(prediction)} "
        f"[{prediction[0]:0.2f}, {prediction[1]:0.2f}]"
    )
    plt.axis("off")
    plt.show()

# Test Wing Classifier Models

Load all models

In [ ]:
from hdr_hybrid_butterflies.model import WingCNNClasifier

model_dict_wing = {}

# upper wing models
for class_number in range(14):
    name = f"wing_subspecies_model_{class_number:02d}"
    print(f"Loading model: {name}")
    model_dict_wing[name] = WingCNNClasifier(
        image_size=image_processor.output_dim,
        num_classes=2,
        name=name,
        verbose=False,
    )
    model_dict_wing[name].load_weights(
        os.path.join(
            model_dir,
            f"wing_subspecies_model_{class_number:02d}",
            "model.weights.h5",
        )
    )

model_dict_wing

In [ ]:
from hdr_hybrid_butterflies.data_handler import WingSegmentDataHandler

image_processor_wing = ImageProcessor(
    # segment_classifier=model,
    # cnn_segmenter=cnn_segmenter,
    # cnn_segmenter_processor=cnn_segmenter_processor,
)
data_handler_wing = WingSegmentDataHandler(
    meta_data_path=meta_data_path,
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    image_processor=image_processor_wing,
)

In [ ]:
predictions_wings_list = []
mask_non_hybrid = data_handler.df_meta["hybrid_stat"] == "non-hybrid"
df_meta_non_hybrid = data_handler.df_meta[mask_non_hybrid]
for camid in tqdm(df_meta_non_hybrid["CAMID"], total=np.sum(mask_non_hybrid)):
    df_segments_upper = data_handler_wing.load_df_meta_segments_for_camid(
        camid, labels=["upper"]
    )
    df_segments_lower = data_handler_wing.load_df_meta_segments_for_camid(
        camid, labels=["lower"]
    )

    # if len(df_segments) > 4:
    #     print(df_segments)
    #     break
    if len(df_segments_upper) not in [1, 2]:
        print(f"Camid: {camid} | Upper Segments: {len(df_segments_upper)}")
    if len(df_segments_lower) not in [1, 2]:
        print(f"Camid: {camid} | Lower Segments: {len(df_segments_lower)}")

    if len(df_segments_upper) > 0:
        # load segments for upper
        segments_upper = []
        for idx, row in df_segments_upper.iterrows():
            segment, _ = data_handler_wing.load_by_name(row["filename"])
            segment = image_processor_wing.augment_image(
                segment, apply_augmentations=False
            )[0]
            segments_upper.append(segment)

    if len(df_segments_lower) > 0:
        # load segments for lower
        segments_lower = []
        for idx, row in df_segments_lower.iterrows():
            segment, _ = segment_data_handler.load_by_name(row["filename"])
            segment = image_processor_wing.augment_image(
                segment, apply_augmentations=False
            )[0]
            segments_lower.append(segment)

    if len(segments_upper) > 2:
        segments_upper = segments_upper[:2]
    if len(segments_lower) > 2:
        segments_lower = segments_lower[:2]

    while len(segments_upper) != 2:
        segments_upper.append(np.zeros_like(segments_upper[0]))
    while len(segments_lower) != 2:
        segments_lower.append(np.zeros_like(segments_lower[0]))

    segments_upper = np.stack(segments_upper, axis=0)
    segments_lower = np.stack(segments_lower, axis=0)

    wing = np.concatenate(
        [segments_upper, segments_lower],
        axis=0,
    )[None, ...]

    # get predictions
    predictions_i = []
    for name in sorted(model_dict_wing.keys()):
        model_wing_i = model_dict_wing[name]
        predictions_i.append(model_wing_i.probabilities(wing))

    predictions_wings_list.append(predictions_i)

predictions_wings = np.array(predictions_wings_list)
predictions_wings = np.squeeze(predictions_wings, axis=2)

In [ ]:
predictions_wings_list = []
mask_hybrid = data_handler.df_meta["hybrid_stat"] == "hybrid"
df_meta_hybrid = data_handler.df_meta[mask_hybrid]
for camid in tqdm(df_meta_hybrid["CAMID"], total=np.sum(mask_hybrid)):
    df_segments_upper = data_handler_wing.load_df_meta_segments_for_camid(
        camid, labels=["upper"]
    )
    df_segments_lower = data_handler_wing.load_df_meta_segments_for_camid(
        camid, labels=["lower"]
    )

    # if len(df_segments) > 4:
    #     print(df_segments)
    #     break
    if len(df_segments_upper) not in [1, 2]:
        print(f"Camid: {camid} | Upper Segments: {len(df_segments_upper)}")
    if len(df_segments_lower) not in [1, 2]:
        print(f"Camid: {camid} | Lower Segments: {len(df_segments_lower)}")

    if len(df_segments_upper) > 0:
        # load segments for upper
        segments_upper = []
        for idx, row in df_segments_upper.iterrows():
            segment, _ = data_handler_wing.load_by_name(row["filename"])
            segment = image_processor_wing.augment_image(
                segment, apply_augmentations=False
            )[0]
            segments_upper.append(segment)

    if len(df_segments_lower) > 0:
        # load segments for lower
        segments_lower = []
        for idx, row in df_segments_lower.iterrows():
            segment, _ = segment_data_handler.load_by_name(row["filename"])
            segment = image_processor_wing.augment_image(
                segment, apply_augmentations=False
            )[0]
            segments_lower.append(segment)

    if len(segments_upper) > 2:
        segments_upper = segments_upper[:2]
    if len(segments_lower) > 2:
        segments_lower = segments_lower[:2]

    while len(segments_upper) != 2:
        segments_upper.append(np.zeros_like(segments_upper[0]))
    while len(segments_lower) != 2:
        segments_lower.append(np.zeros_like(segments_lower[0]))

    segments_upper = np.stack(segments_upper, axis=0)
    segments_lower = np.stack(segments_lower, axis=0)

    wing = np.concatenate(
        [segments_upper, segments_lower],
        axis=0,
    )[None, ...]

    # get predictions
    predictions_i = []
    for name in sorted(model_dict_wing.keys()):
        model_wing_i = model_dict_wing[name]
        predictions_i.append(model_wing_i.probabilities(wing))

    predictions_wings_list.append(predictions_i)

predictions_wings_hybrid = np.array(predictions_wings_list)
predictions_wings_hybrid = np.squeeze(predictions_wings_hybrid, axis=2)

In [ ]:
predictions_wings.shape

In [ ]:
mask_dict = {}
for i in range(14):
    mask_dict[f"class_{i:02d}"] = df_meta_non_hybrid["subspecies"] == i


# test set
mask_dict["test"] = np.zeros(len(df_meta_non_hybrid), dtype=bool)
for idx, (_, row) in enumerate(df_meta_non_hybrid.iterrows()):
    if (
        row["CAMID"]
        in data_handler_wing.df_meta.iloc[data_handler_wing.n_samples_train :][
            "CAMID"
        ].values
    ):
        mask_dict["test"][idx] = True

mask_dict.keys()

In [ ]:
predictions_wings.shape

In [ ]:
bins = np.linspace(0, 1, 100)
fig, axes = plt.subplots(5, 3, figsize=(13, 13))

for idx, ax in enumerate(axes.flatten()):
    if idx >= 14:
        break
    ax.hist(
        predictions_wings[mask_dict[f"class_{idx:02d}"], idx, 1],
        bins=bins,
        histtype="step",
        label=f"Wing {idx} | pred 1",
    )
    ax.hist(
        predictions_wings[~mask_dict[f"class_{idx:02d}"], idx, 1],
        bins=bins,
        histtype="step",
        label=f"Wing {idx} [non] | pred 1",
    )
    ax.hist(
        predictions_wings[
            mask_dict[f"class_{idx:02d}"] & mask_dict["test"], idx, 1
        ],
        bins=bins,
        histtype="step",
        ls="--",
        label=f"[TEST] Wing {idx} | pred 1",
    )
    ax.hist(
        predictions_wings[
            ~mask_dict[f"class_{idx:02d}"] & mask_dict["test"], idx, 1
        ],
        bins=bins,
        histtype="step",
        ls="--",
        label=f"[TEST] Wing {idx} [non] | pred 1",
    )

    # plot hybrid
    ax.hist(
        predictions_wings_hybrid[:, idx, 1],
        bins=bins,
        histtype="step",
        color="0.0",
        ls=":",
        lw=3,
        label=f"Hybrid Wing {idx} | pred 1",
    )

    ax.set_title(f"Wing | Class {idx}")
    ax.set_yscale("log")
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Count")
    ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "wing_predictions.png"))

In [ ]:
def get_wing_bkg_distributions(
    predictions,
    df_meta_non_hybrid,
):
    n_classes = predictions.shape[1]
    distributions_bkg = {}
    distributions_sig = {}
    for class_idx in range(n_classes):
        mask = mask_dict[f"class_{class_idx:02d}"]
        predictions_sig = predictions[mask][:, class_idx, 1]
        predictions_bkg = predictions[~mask][:, class_idx, 1]

        distributions_sig[class_idx] = np.sort(predictions_sig, axis=0)
        distributions_bkg[class_idx] = np.sort(predictions_bkg, axis=0)
    return distributions_bkg, distributions_sig


wing_bkg_distributions, wing_sig_distributions = get_wing_bkg_distributions(
    predictions_wings,
    df_meta_non_hybrid,
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

bins = np.linspace(0, 1, 20)
# for key, values in wing_sig_distributions.items():
#     ax.hist(
#         values,
#         label=f"Class {key} | Signal",
#         histtype="step",
#         alpha=0.5,
#         bins=bins,
#     )
for key, values in wing_bkg_distributions.items():
    if key != 8:
        continue
    ax.hist(
        values,
        label=f"Class {key} | Background",
        alpha=0.5,
        bins=bins,
    )
ax.set_yscale("log")
ax.set_xlabel("Prediction")
ax.set_ylabel("Counts")
ax.legend()

In [ ]:
predictions_wings.shape

In [ ]:
def compute_class_p_values(
    predictions,
    wing_bkg_distributions,
):
    """Compute p-values for each class based on the background distribution.

    Parameters
    ----------
    predictions : np.ndarray
        Array of shape (n_samples, n_classes, 2) containing the predictions.
    wing_bkg_distributions : dict
        Dictionary containing the background distributions for each class.

    Returns
    -------
    p_values : np.ndarray
        Array of shape (n_samples, n_classes) containing the p-values
        for each sample and class.
    """
    n_samples, n_classes, _ = predictions.shape
    p_values = np.zeros((n_samples, n_classes))
    for class_idx in range(n_classes):
        bkg_dist = wing_bkg_distributions[class_idx]
        p_values_i = 1.0 - np.searchsorted(
            bkg_dist, predictions[:, class_idx, 1]
        ) / len(bkg_dist)
        p_values[:, class_idx] = p_values_i
    return p_values


def compute_anomaly_ts(class_p_values, exclude_classes=[6, 8, 9]):
    """Compute the test statistic for the anomaly detection.

    Parameters
    ----------
    class_p_values : np.ndarray
        Array of shape (n_samples, n_classes) containing the p-values
        for each sample and class.
    exclude_classes : list, optional
        List of class indices to exclude from the anomaly test statistic.

    Returns
    -------
    anomaly_ts : np.ndarray
        Array of shape (n_samples,) containing the anomaly test statistic.
    """
    include_classes = np.ones(class_p_values.shape[1], dtype=bool)
    if exclude_classes is not None:
        include_classes[exclude_classes] = False

    class_p_values = class_p_values[:, include_classes]

    # shape (n_samples, n_classes)
    p_sig = 1 - np.clip(class_p_values, 1e-4, 1)

    # Anomalies are special, because they should have at least two high p-values
    # Sort p-values in descending order
    p_sig_sorted = np.sort(p_sig, axis=-1)[:, ::-1]

    # return np.log(p_sig_sorted[:, 1])

    # # Compute the anomaly test statistic excluding the highest p-value
    # return np.sum(np.log(p_sig_sorted[:, 1:]), axis=-1)

    # Compute the anomaly test statistic based on the two highest p-values
    return np.sum(np.log(p_sig_sorted[:, :2]), axis=-1)

    return np.sum(np.log(np.clip(1 - class_p_values, 1e-3, 1)), axis=-1)


# shape (n_samples, n_classes)
p_values_wings = compute_class_p_values(
    predictions=predictions_wings,
    wing_bkg_distributions=wing_bkg_distributions,
)
p_values_wings_hybrid = compute_class_p_values(
    predictions=predictions_wings_hybrid,
    wing_bkg_distributions=wing_bkg_distributions,
)

bkg_anomoly_ts = compute_anomaly_ts(p_values_wings)
sig_anomoly_ts_ = compute_anomaly_ts(p_values_wings_hybrid)

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.hist(
    -bkg_anomoly_ts,
    bins=np.logspace(-4, 2, 1000),
    histtype="step",
    label="Background",
)
ax.hist(
    -sig_anomoly_ts_,
    bins=np.logspace(-4, 2, 1000),
    histtype="step",
    label="Signal",
)
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_xlabel(" - Anomaly Test Statistic")
ax.set_ylabel("Counts")

p_values_wings.shape, bkg_anomoly_ts.shape

In [ ]:
xscale = "log"
# xscale = "linear"

fig, axes = plt.subplots(3, 5, figsize=(20, 10))
axes_flat = axes.flatten()

n_classes = predictions_wings.shape[1]
if xscale == "log":
    bins = np.logspace(-4, 0, 50)
else:
    bins = np.linspace(0, 1, 50)

for class_idx in range(n_classes):
    ax = axes_flat[class_idx]

    mask = mask_dict[f"class_{class_idx:02d}"]
    ax.hist(
        p_values_wings[~mask, class_idx],
        bins=bins,
        alpha=0.5,
        linewidth=2,
        label=f"Subspecies {class_idx} [N={np.sum(~mask)}]",
    )
    ax.hist(
        p_values_wings_hybrid[:, class_idx],
        bins=bins,
        alpha=0.5,
        linewidth=2,
        label=f"Hybrid [N={len(p_values_wings_hybrid)}]",
    )
    for subspecies in range(n_classes):
        # if subspecies != class_idx:
        #     continue
        mask = mask_dict[f"class_{subspecies:02d}"]
        label = None
        if (
            # np.median(p_values_wings[mask, class_idx]) > 1e-2 or
            subspecies
            == class_idx
        ):
            label = f"Subspecies {subspecies} [N={np.sum(mask)}]"

        values = np.clip(p_values_wings[mask, class_idx], 1e-4, float("inf"))
        if subspecies == class_idx:
            ax.hist(
                values,
                bins=bins,
                histtype="step",
                color="0.0",
                ls="-",
                label=f"Signal Class [N={np.sum(mask)}]",
            )
        else:
            ax.hist(
                values,
                bins=bins,
                histtype="step",
                ls="--",
                label=label,
            )
    ax.set_title("Class Background P-value Distributions")
    ax.set_yscale("log")
    ax.set_xlabel("P-value")
    ax.set_ylabel("Count")
    ax.legend()
    ax.set_xscale(xscale)
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "wing_background_pval_distributions.png"))

# Test Models

Load all models

In [ ]:
from hdr_hybrid_butterflies.model import CNNClasifier

model_dict = {}

# upper wing models
for feature_number in range(12):
    name = f"upper_wing_feature_{feature_number:02d}"
    model_dict[name] = CNNClasifier(
        image_size=image_processor.output_dim,
        num_classes=2,
        name=name,
        verbose=False,
    )
    model_dict[name].load_weights(
        os.path.join(
            model_dir, f"upper_model_{feature_number:02d}", "model.weights.h5"
        )
    )

# lower wing models
for feature_number in range(4):
    name = f"lower_wing_feature_{feature_number:02d}"
    model_dict[name] = CNNClasifier(
        image_size=image_processor.output_dim,
        num_classes=2,
        name=name,
        verbose=False,
    )
    model_dict[name].load_weights(
        os.path.join(
            model_dir, f"lower_model_{feature_number:02d}", "model.weights.h5"
        )
    )
model_dict

In [ ]:
model_keys_upper = [
    model_name for model_name in model_dict.keys() if "upper" in model_name
]
model_keys_lower = [
    model_name for model_name in model_dict.keys() if "lower" in model_name
]

predictions_list_upper = []
predictions_list_lower = []
mask_non_hybrid = data_handler.df_meta["hybrid_stat"] == "non-hybrid"
df_meta_non_hybrid = data_handler.df_meta[mask_non_hybrid]
for camid in tqdm(df_meta_non_hybrid["CAMID"], total=np.sum(mask_non_hybrid)):
    df_segments_upper = segment_data_handler.load_df_meta_segments_for_camid(
        camid, labels=["upper"]
    )
    df_segments_lower = segment_data_handler.load_df_meta_segments_for_camid(
        camid, labels=["lower"]
    )

    # if len(df_segments) > 4:
    #     print(df_segments)
    #     break
    if len(df_segments_upper) not in [1, 2]:
        print(f"Camid: {camid} | Upper Segments: {len(df_segments_upper)}")
    if len(df_segments_lower) not in [1, 2]:
        print(f"Camid: {camid} | Lower Segments: {len(df_segments_lower)}")

    if len(df_segments_upper) > 0:
        # load segments for upper
        segments_upper = []
        for idx, row in df_segments_upper.iterrows():
            segment, _ = segment_data_handler.load_by_name(row["filename"])
            segment = image_processor.augment_image(
                segment, apply_augmentations=False
            )
            segments_upper.append(segment)
        segments_upper = np.stack(segments_upper, axis=0)

        # get predictions for upper
        predictions_upper = np.zeros(
            (len(model_keys_upper), len(df_segments_upper), 2)
        )
        for idx, model_name in enumerate(model_keys_upper):
            model = model_dict[model_name]
            predictions_upper[idx] = model.probabilities(segments_upper)
        predictions_upper = np.max(predictions_upper, axis=1)
    else:
        predictions_upper = np.zeros((len(model_keys_upper), 2))

    if len(df_segments_lower) > 0:
        # load segments for lower
        segments_lower = []
        for idx, row in df_segments_lower.iterrows():
            segment, _ = segment_data_handler.load_by_name(row["filename"])
            segment = image_processor.augment_image(
                segment, apply_augmentations=False
            )
            segments_lower.append(segment)
        segments_lower = np.stack(segments_lower, axis=0)

        # get predictions for lower
        predictions_lower = np.zeros(
            (len(model_keys_lower), len(df_segments_lower), 2)
        )
        for idx, model_name in enumerate(model_keys_lower):
            model = model_dict[model_name]
            predictions_lower[idx] = model.probabilities(segments_lower)
        predictions_lower = np.max(predictions_lower, axis=1)
    else:
        predictions_lower = np.zeros((len(model_keys_lower), 2))

    predictions_list_upper.append(predictions_upper)
    predictions_list_lower.append(predictions_lower)

predictions_upper = np.stack(predictions_list_upper, axis=0)
predictions_lower = np.stack(predictions_list_lower, axis=0)

In [ ]:
predictions_upper.shape, predictions_lower.shape

In [ ]:
from hdr_hybrid_butterflies.data_handler import (
    UpperWingDataHandler,
    LowerWingDataHandler,
)

upper_wing_data_handler = UpperWingDataHandler(
    meta_data_path=meta_data_path,
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    image_processor=image_processor,
)

lower_wing_data_handler = LowerWingDataHandler(
    meta_data_path=meta_data_path,
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    image_processor=image_processor,
)

# upper_wing_data_handler_hybrid = UpperWingDataHandler(
#     meta_data_path=meta_data_path,
#     data_dir_upper=os.path.join(
#         segment_training_dir, "manual", "upper_wing_manual"
#     ),
#     image_processor=image_processor,
#     skip_hybrid=False,
# )

# lower_wing_data_handler_hybrid = LowerWingDataHandler(
#     meta_data_path=meta_data_path,
#     data_dir_lower=os.path.join(
#         segment_training_dir, "manual", "lower_wing_manual"
#     ),
#     image_processor=image_processor,
#     skip_hybrid=False,
# )


segment_data_handler_hybrid = SegmentDataHandler(
    meta_data_path=meta_data_path,
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    data_dir_noise=os.path.join(
        segment_training_dir, "manual", "noise_manual"
    ),
    image_processor=image_processor,
    skip_hybrid=False,
)

In [ ]:
predictions_hybrid_list = []

mask_non_hybrid = data_handler.df_meta["hybrid_stat"] == "non-hybrid"
mask_hybrid = ~mask_non_hybrid
df_meta_hybrid = data_handler.df_meta[mask_hybrid]
for camid in tqdm(df_meta_hybrid["CAMID"], total=np.sum(mask_hybrid)):
    df_segments_upper = (
        segment_data_handler_hybrid.load_df_meta_segments_for_camid(
            camid, labels=["upper"]
        )
    )
    df_segments_lower = (
        segment_data_handler_hybrid.load_df_meta_segments_for_camid(
            camid, labels=["lower"]
        )
    )

    if len(df_segments_upper) not in [1, 2]:
        print(f"Camid: {camid} | Upper Segments: {len(df_segments_upper)}")
    if len(df_segments_lower) not in [1, 2]:
        print(f"Camid: {camid} | Lower Segments: {len(df_segments_lower)}")

    if len(df_segments_upper) > 0:
        # load segments for upper
        segments_upper = []
        for idx, row in df_segments_upper.iterrows():
            segment, _ = segment_data_handler_hybrid.load_by_name(
                row["filename"]
            )
            segment = image_processor.augment_image(
                segment, apply_augmentations=False
            )
            segments_upper.append(segment)
        segments_upper = np.stack(segments_upper, axis=0)

        # get predictions for upper
        predictions_upper_h = np.zeros(
            (len(model_keys_upper), len(df_segments_upper), 2)
        )
        for idx, model_name in enumerate(model_keys_upper):
            model = model_dict[model_name]
            logits = model.call(segments_upper)
            predictions_upper_h[idx] = model.logits2probs(logits)
        predictions_upper_h = np.max(predictions_upper_h, axis=1)
    else:
        predictions_upper_h = np.zeros((len(model_keys_upper), 2))

    if len(df_segments_lower) > 0:
        # load segments for lower
        segments_lower = []
        for idx, row in df_segments_lower.iterrows():
            segment, _ = segment_data_handler_hybrid.load_by_name(
                row["filename"]
            )
            segment = image_processor.augment_image(
                segment, apply_augmentations=False
            )
            segments_lower.append(segment)
        segments_lower = np.stack(segments_lower, axis=0)

        # get predictions for lower
        predictions_lower_h = np.zeros(
            (len(model_keys_lower), len(df_segments_lower), 2)
        )
        for idx, model_name in enumerate(model_keys_lower):
            model = model_dict[model_name]
            logits = model.call(segments_lower)
            predictions_lower_h[idx] = model.logits2probs(logits)
        predictions_lower_h = np.max(predictions_lower_h, axis=1)
    else:
        predictions_lower_h = np.zeros((len(model_keys_lower), 2))

    predictions_hybrid_list.append(
        np.concatenate([predictions_lower_h, predictions_upper_h], axis=0)
    )

predictions_hybrid = np.stack(predictions_hybrid_list, axis=0)

In [ ]:
mask_dict = {}
for i in range(12):
    mask_dict[f"upper_{i:02d}"] = df_meta_non_hybrid["subspecies"].isin(
        upper_wing_data_handler.feature_definitions[i]
    )

for i in range(4):
    mask_dict[f"lower_{i:02d}"] = df_meta_non_hybrid["subspecies"].isin(
        lower_wing_data_handler.feature_definitions[i]
    )


# test set
mask_dict["test"] = np.zeros(len(df_meta_non_hybrid), dtype=bool)
for idx, (_, row) in enumerate(df_meta_non_hybrid.iterrows()):
    if (
        row["CAMID"]
        in lower_wing_data_handler.df_meta.iloc[
            lower_wing_data_handler.n_samples_train :
        ]["CAMID"].values
    ):
        mask_dict["test"][idx] = True
    elif (
        row["CAMID"]
        in upper_wing_data_handler.df_meta.iloc[
            upper_wing_data_handler.n_samples_train :
        ]["CAMID"].values
    ):
        mask_dict["test"][idx] = True

mask_dict.keys()

In [ ]:
predictions_upper.shape, predictions_lower.shape

In [39]:
bins = np.linspace(0, 1, 100)
fig, axes = plt.subplots(4, 3, figsize=(13, 13))

for idx, ax in enumerate(axes.flatten()):
    # ax.hist(
    #     predictions_upper[mask_dict[f"upper_{idx:02d}"], idx, 0],
    #     bins=bins,
    #     histtype="step",
    #     label=f"Upper Wing {idx} | pred 0",
    # )
    # ax.hist(
    #     predictions_upper[~mask_dict[f"upper_{idx:02d}"], idx, 0],
    #     bins=bins,
    #     histtype="step",
    #     label=f"Upper Wing {idx} [non] | pred 0",
    # )
    ax.hist(
        predictions_upper[mask_dict[f"upper_{idx:02d}"], idx, 1],
        bins=bins,
        histtype="step",
        label=f"Upper Wing {idx} | pred 1",
    )
    ax.hist(
        predictions_upper[~mask_dict[f"upper_{idx:02d}"], idx, 1],
        bins=bins,
        histtype="step",
        label=f"Upper Wing {idx} [non] | pred 1",
    )
    ax.hist(
        predictions_upper[
            mask_dict[f"upper_{idx:02d}"] & mask_dict["test"], idx, 1
        ],
        bins=bins,
        histtype="step",
        ls="--",
        label=f"[TEST] Upper Wing {idx} | pred 1",
    )
    ax.hist(
        predictions_upper[
            ~mask_dict[f"upper_{idx:02d}"] & mask_dict["test"], idx, 1
        ],
        bins=bins,
        histtype="step",
        ls="--",
        label=f"[TEST] Upper Wing {idx} [non] | pred 1",
    )
    ax.set_title(f"Upper Wing | Feature {idx}")
    ax.set_yscale("log")
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Count")
    ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "upper_wing_predictions.png"))

In [40]:
bins = np.linspace(0, 1, 100)
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for idx, ax in enumerate(axes.flatten()):
    # ax.hist(
    #     predictions_lower[mask_dict[f"lower_{idx:02d}"], idx, 0],
    #     bins=bins,
    #     histtype="step",
    #     label=f"Lower Wing {idx} | pred 0",
    # )
    # ax.hist(
    #     predictions_lower[~mask_dict[f"lower_{idx:02d}"], idx, 0],
    #     bins=bins,
    #     histtype="step",
    #     label=f"Lower Wing {idx} [non] | pred 0",
    # )
    ax.hist(
        predictions_lower[mask_dict[f"lower_{idx:02d}"], idx, 1],
        bins=bins,
        histtype="step",
        label=f"Lower Wing {idx} | pred 1",
    )
    ax.hist(
        predictions_lower[~mask_dict[f"lower_{idx:02d}"], idx, 1],
        bins=bins,
        histtype="step",
        label=f"Lower Wing {idx} [non] | pred 1",
    )
    ax.hist(
        predictions_lower[
            mask_dict[f"lower_{idx:02d}"] & mask_dict["test"], idx, 1
        ],
        bins=bins,
        histtype="step",
        ls="--",
        label=f"[TEST] Lower Wing {idx} | pred 1",
    )
    ax.hist(
        predictions_lower[
            ~mask_dict[f"lower_{idx:02d}"] & mask_dict["test"], idx, 1
        ],
        bins=bins,
        histtype="step",
        ls="--",
        label=f"[TEST] Lower Wing {idx} [non] | pred 1",
    )
    ax.set_title(f"Lower Wing | Feature {idx}")
    ax.set_yscale("log")
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Count")
    ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "lower_wing_predictions.png"))

In [ ]:
predictions_upper.shape, predictions_lower.shape

## Build ts distributions

In [ ]:
def get_subspecies_feature_definitions(
    upper_wing_data_handler, lower_wing_data_handler
):
    subspecies_feature_definitions = dict(
        lower_wing_data_handler.feature_definitions
    )
    for i in range(12):
        subspecies_feature_definitions[
            i + 4
        ] = upper_wing_data_handler.feature_definitions[i]
    return subspecies_feature_definitions


subspecies_feature_definitions = get_subspecies_feature_definitions(
    upper_wing_data_handler=upper_wing_data_handler,
    lower_wing_data_handler=lower_wing_data_handler,
)
subspecies_features = {idx: [] for idx in range(14)}
for key, value in subspecies_feature_definitions.items():
    for sub in value:
        subspecies_features[sub].append(key)
subspecies_feature_definitions

In [ ]:
subspecies_features

In [ ]:
def get_feature_distributions(
    predictions_upper,
    predictions_lower,
    df_meta_non_hybrid,
):
    n_features = 16
    distributions_bkg = {}
    distributions_sig = {}
    for feature_idx in range(n_features):
        if feature_idx < 4:
            mask = mask_dict[f"lower_{feature_idx:02d}"]
            predictions_sig = predictions_lower[mask][:, feature_idx, 1]
            predictions_bkg = predictions_lower[~mask][:, feature_idx, 1]
        else:
            mask = mask_dict[f"upper_{feature_idx-4:02d}"]
            predictions_sig = predictions_upper[mask][:, feature_idx - 4, 1]
            predictions_bkg = predictions_upper[~mask][:, feature_idx - 4, 1]
        predictions_sig = np.sort(predictions_sig, axis=0)
        predictions_bkg = np.sort(predictions_bkg, axis=0)
        distributions_sig[feature_idx] = predictions_sig
        distributions_bkg[feature_idx] = predictions_bkg
    return distributions_bkg, distributions_sig


(
    feature_bkg_distributions,
    feature_sig_distributions,
) = get_feature_distributions(
    predictions_upper=predictions_upper,
    predictions_lower=predictions_lower,
    df_meta_non_hybrid=df_meta_non_hybrid,
)

print("Feature signal distributions")
for key, values in feature_sig_distributions.items():
    print(
        f"Feature idx: {key} | {values.shape} | min: {np.min(values):3.6e} | max: {np.max(values):3.6e}"
    )
print()
print("Feature background distributions")
for key, values in feature_bkg_distributions.items():
    print(
        f"Feature idx: {key} | {values.shape} | min: {np.min(values):3.6e} | max: {np.max(values):3.6e}"
    )

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

bins = np.linspace(0, 1, 20)
for key, values in feature_sig_distributions.items():
    ax.hist(
        values,
        label=f"Feature {key} | Signal",
        histtype="step",
        alpha=0.5,
        bins=bins,
    )
for key, values in feature_bkg_distributions.items():
    ax.hist(
        values,
        label=f"Feature {key} | Background",
        alpha=0.5,
        bins=bins,
    )
ax.set_yscale("log")
ax.set_xlabel("Prediction")
ax.set_ylabel("Counts")
ax.legend()

In [46]:
def feature_ts(pval_feature, threshold=0.1):
    """Calculate the ts-value for a given feature p-value.

    Parameters
    ----------
    pval_feature : np.array
        The p-value of a certain feature for a given subspecies.
    threshold : float
        The threshold for the p-value to be considered significant.
        Anything over this threshold will not contribute to the
        calculation of the ts-value.

    Returns
    -------
    ts : float
        The ts-value for the given feature.
    """
    return np.where(
        pval_feature < threshold,
        (threshold - pval_feature) / threshold,
        0,
    )


def calibrate_feature_probabilities(
    predictions,
    feature_bkg_distributions,
    feature_sig_distributions,
):
    """Calibrate the feature probabilities.

    Parameters
    ----------
    predictions : np.array
        The predictions for the features.
        This array should be concatenated from the predictions
        of the lower (first) and upper (second) wing models.
        Last dimension is the probability for the feature not being
        present (index 0) and the probability for the feature being
        present (index 1).
        Shape: (n_samples, n_features, 2)
    feature_bkg_distributions : dict
        The prediction value distributions for each feature
        on background samples where the feature is not present.
    feature_sig_distributions : dict
        The prediction value distributions for each feature
        on signal samples where the feature is present

    Returns
    -------
    calibrated_predictions : np.array
        The calibrated predictions for the features.
        Shape: (n_samples, n_features)
    """
    calibrated_predictions = np.zeros_like(predictions)
    for feature_idx in range(predictions.shape[1]):
        bkg_values = feature_bkg_distributions[feature_idx]
        sig_values = feature_sig_distributions[feature_idx]

        pval_present = np.searchsorted(
            bkg_values,
            predictions[:, feature_idx, 1],
            side="right",
        ) / len(bkg_values)
        pval_not_present = np.searchsorted(
            sig_values,
            predictions[:, feature_idx, 0],
            side="right",
        ) / len(sig_values)

        calibrated_predictions[:, feature_idx, 0] = pval_not_present
        calibrated_predictions[:, feature_idx, 1] = pval_present

    return calibrated_predictions


def compute_feature_ts(
    predictions,
    feature_bkg_distributions,
    feature_sig_distributions,
    subspecies_feature_definitions,
    threshold=0.1,
):
    """Compute the ts-values for all features.

    Parameters
    ----------
    predictions : np.array
        The predictions for the features.
        This array should be concatenated from the predictions
        of the lower (first) and upper (second) wing models and
        only take the second column (i.e. the probability of
        the feature being present).
        Shape: (n_samples, n_features)
    feature_bkg_distributions : dict
        The prediction value distributions for each feature
        on background samples where the feature is not present.
    feature_sig_distributions : dict
        The prediction value distributions for each feature
        on signal samples where the feature is present
    subspecies_feature_definitions : dict[list[int]]
        The feature definitions for each subspecies.
        The keys are the subspecies and the values are lists
        of feature indices.
        This dictionary should be created from the data handler
        for the lower and upper wing models.
    threshold : float
        The threshold for the p-value to be considered significant.
        Anything over this threshold will not contribute to the
        calculation of the ts-value.

    Returns
    -------
    ts_values : np.array
        The ts-values for all features.
        Shape: (n_samples, n_subspecies, n_features)
    """
    n_subspecies = 14
    n_features = predictions.shape[1]
    ts_values = np.zeros((len(predictions), n_subspecies, n_features))

    for subspecies_idx in range(n_subspecies):
        for feature_idx in range(n_features):
            # p-value based on rejecting the signal-only hypothesis
            if subspecies_idx in subspecies_feature_definitions[feature_idx]:
                pval_feature = np.searchsorted(
                    feature_sig_distributions[feature_idx],
                    predictions[:, feature_idx],
                    side="right",
                ) / len(feature_sig_distributions[feature_idx])

            # p-value based on rejecting the bkg-only hypothesis
            else:
                pval_feature = np.searchsorted(
                    feature_bkg_distributions[feature_idx],
                    predictions[:, feature_idx],
                    side="right",
                ) / len(feature_bkg_distributions[feature_idx])
                pval_feature = 1 - pval_feature

            ts_values[:, subspecies_idx, feature_idx] = feature_ts(
                pval_feature, threshold=threshold
            )
    return ts_values


def compute_class_pval(
    predictions,
    class_bkg_ts_distributions,
    feature_bkg_distributions,
    feature_sig_distributions,
    subspecies_feature_definitions,
):
    """Compute the p-values for each subspecies class.

    Parameters
    ----------
    predictions : np.array
        The predictions for the features.
        This array should be concatenated from the predictions
        of the lower (first) and upper (second) wing models and
        only take the second column (i.e. the probability of
        the feature being present).
        Shape: (n_samples, n_features)
    class_bkg_ts_distributions : dict
        The background ts-values for each class.
        The keys are the class names and the values are the
        ts-values for the class.
        Shape: (n_samples, n_subspecies)
    feature_bkg_distributions : dict
        The prediction value distributions for each feature
        on background samples where the feature is not present.
    feature_sig_distributions : dict
        The prediction value distributions for each feature
        on signal samples where the feature is present
    subspecies_feature_definitions : dict[list[int]]
        The feature definitions for each subspecies.
        The keys are the subspecies and the values are lists
        of feature indices.

    Returns
    -------
    class_pvals : np.array
        The p-values for all subspecies classes.
        Shape: (n_samples, n_subspecies)
    """
    n_samples = len(predictions)
    n_subspecies = len(class_ts_distributions)
    class_pvals = np.zeros((n_samples, n_subspecies))

    # shape: (n_samples, n_subspecies, n_features)
    feature_ts = compute_feature_ts(
        predictions=predictions,
        feature_bkg_distributions=feature_bkg_distributions,
        feature_sig_distributions=feature_sig_distributions,
        subspecies_feature_definitions=subspecies_feature_definitions,
    )

    # shape: (n_samples, n_subspecies)
    subspecies_ts = np.sum(feature_ts, axis=2)

    subspecies_pval = np.zeros_like(subspecies_ts)
    for subspecies_idx in range(n_subspecies):
        subspecies_pval[:, subspecies_idx] = np.searchsorted(
            class_bkg_ts_distributions[subspecies_idx],
            subspecies_ts[:, subspecies_idx],
            side="right",
        ) / len(class_bkg_ts_distributions[subspecies_idx])
        subspecies_pval = 1 - subspecies_pval
    return subspecies_pval


def compute_pval(
    predictions,
    subspecies_bkg_pval_distribution,
    class_bkg_ts_distributions,
    feature_bkg_distributions,
    feature_sig_distributions,
    subspecies_feature_definitions,
    reduce_func=lambda x: np.max(x, axis=1),
):
    """Compute the p-value for the given predictions.

    Parameters
    ----------
    predictions : np.array
        The predictions for the features.
        This array should be concatenated from the predictions
        of the lower (first) and upper (second) wing models and
        only take the second column (i.e. the probability of
        the feature being present).
        Shape: (n_samples, n_features)
    subspecies_bkg_pval_distribution : np.array
        The p-value distribution for the subspecies background.
        Shape: (n_samples, n_subspecies)
    class_bkg_ts_distributions : dict
        The background ts-values for each class.
        The keys are the class names and the values are the
        ts-values for the class.
        Shape: (n_samples, n_subspecies)
    feature_bkg_distributions : dict
        The prediction value distributions for each feature
        on background samples where the feature is not present.
    feature_sig_distributions : dict
        The prediction value distributions for each feature
        on signal samples where the feature is present
    subspecies_feature_definitions : dict[list[int]]
        The feature definitions for each subspecies.
        The keys are the subs
    reduce_func : function
        The function to reduce the p-values for the subspecies.
        Default: np.max, i.e. we need to reject each subspecies hypothesis
        and our overall result is based on the subspecies that we have
        the weakest evidence against.

    Returns
    -------
    pval : float
        The p-value for the given predictions.
    """
    pval_subspecies = compute_class_pval(
        predictions=predictions,
        class_bkg_ts_distributions=class_bkg_ts_distributions,
        feature_bkg_distributions=feature_bkg_distributions,
        feature_sig_distributions=feature_sig_distributions,
        subspecies_feature_definitions=subspecies_feature_definitions,
    )

    pval_subspecies = reduce_func(pval_subspecies)
    pval_bkg_distribution = reduce_func(subspecies_bkg_pval_distribution)
    pval_bkg_distribution = np.sort(pval_bkg_distribution)
    print("pval_subspecies", pval_subspecies.shape, np.sort(pval_subspecies))
    print(
        "pval_bkg_distribution",
        pval_bkg_distribution.shape,
        pval_bkg_distribution,
    )
    # pval = np.sum(pval_subspecies[:, np.newaxis] <= pval_bkg_distribution, axis=1) / len(pval_bkg_distribution)
    pval = np.searchsorted(
        pval_bkg_distribution, pval_subspecies, side="right"
    ) / len(pval_bkg_distribution)
    return pval

In [ ]:
predictions_calibrated = calibrate_feature_probabilities(
    predictions=np.concatenate([predictions_lower, predictions_upper], axis=1),
    feature_bkg_distributions=feature_bkg_distributions,
    feature_sig_distributions=feature_sig_distributions,
)

predictions_hybrid_calibrated = calibrate_feature_probabilities(
    predictions=predictions_hybrid,
    feature_bkg_distributions=feature_bkg_distributions,
    feature_sig_distributions=feature_sig_distributions,
)

bins = np.linspace(0, 1, 10)
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
for feature_idx in range(16):
    ax.hist(
        predictions_calibrated[:, feature_idx, 1],
        label=f"Feature {feature_idx}",
        histtype="step",
        bins=bins,
    )
ax.set_yscale("log")
ax.set_xlabel("Calibrated Prediction")
ax.set_ylabel("Counts")
ax.legend()

In [ ]:
subspecies_features, subspecies_feature_definitions

In [49]:
# import cv2
# # Set up the detector with default parameters.
# blob_detector = cv2.SimpleBlobDetector()

# # Detect blobs.
# keypoints = blob_detector.detect(segment)
# keypoints

In [50]:
import matplotlib

%matplotlib inline

In [ ]:
import albumentations as A


segment, segment_info = segment_data_handler.load_data(100)
segment = image_processor.augment_image(segment, apply_augmentations=False)
mask = segment == 0
segment_cp = segment.copy()
trafo = A.Compose(
    [
        #  A.MaskDropout(p=1.0),
        A.Erasing(p=1.0, fill=0, scale=(0.02, 0.33)),
        #  A.CoarseDropout(
        #      p=1.0,
        #      num_holes_range=[10, 100],
        #      hole_height_range=[20, 200],
        #      hole_width_range=[20, 200],
        #     #  fill="inpaint_telea",
        # ),
    ]
)
segment_cp = trafo(image=segment_cp, mask=mask)["image"]
# segment_cp[mask] = 0
plt.imshow(segment_cp)

In [ ]:
from hdr_hybrid_butterflies.data_handler import WingSegmentDataHandler

wing_data_handler = WingSegmentDataHandler(
    meta_data_path=meta_data_path,
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    image_processor=ImageProcessor(),
)

In [ ]:
wing, row = wing_data_handler(
    sample_random_segments=True, apply_augmentations=True
)
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes_flat = axes.flatten()
for i, segment in enumerate(wing):
    axes_flat[i].imshow(segment)
    axes_flat[i].set_title(f"Segment {i}")
    axes_flat[i].axis("off")
plt.show()

In [ ]:
from hdr_hybrid_butterflies.model import WingCNNClasifier

clf = WingCNNClasifier(num_classes=2, image_size=image_processor.output_dim)

In [ ]:
mask = df_meta_non_hybrid.subspecies == 6
avg_prob = np.mean(predictions_calibrated[mask], axis=0)
((1 - avg_prob[:, 0]) + avg_prob[:, 1]) / 2, avg_prob

In [ ]:
def get_class_ts_distributions(
    predictions_upper,
    predictions_lower,
    df_meta_non_hybrid,
    feature_bkg_distributions,
    feature_sig_distributions,
    subspecies_feature_definitions,
):
    predictions = np.concatenate(
        (
            predictions_lower[:, :, 1],
            predictions_upper[:, :, 1],
        ),
        axis=1,
    )

    n_subspecies = 14
    n_features = 16
    distributions = {}
    for subspecies_idx in range(n_subspecies):
        mask_subspecies = df_meta_non_hybrid["subspecies"] == subspecies_idx

        # shape: (n_samples_per_subspecies, n_features)
        predictions_i = predictions[mask_subspecies]

        # get p-values for each feature
        distributions[subspecies_idx] = compute_feature_ts(
            predictions=predictions_i,
            feature_bkg_distributions=feature_bkg_distributions,
            feature_sig_distributions=feature_sig_distributions,
            subspecies_feature_definitions=subspecies_feature_definitions,
        )
    return distributions


def get_class_bkg_ts_distributions(class_ts_distributions):
    class_bkg_ts_distributions = {}
    for subspecies_idx, ts_values in class_ts_distributions.items():
        class_bkg_ts_distributions[subspecies_idx] = np.sort(
            np.sum(ts_values[:, subspecies_idx], axis=1),
            axis=0,
        )
    return class_bkg_ts_distributions


class_ts_distributions = get_class_ts_distributions(
    predictions_upper=predictions_upper,
    predictions_lower=predictions_lower,
    df_meta_non_hybrid=df_meta_non_hybrid,
    feature_bkg_distributions=feature_bkg_distributions,
    feature_sig_distributions=feature_sig_distributions,
    subspecies_feature_definitions=subspecies_feature_definitions,
)
class_bkg_ts_distributions = get_class_bkg_ts_distributions(
    class_ts_distributions
)


for key, values in class_ts_distributions.items():
    print(f"Subspecies: {key} | {values.shape}")

In [ ]:
predictions = np.concatenate(
    (
        predictions_lower[:, :, 1],
        predictions_upper[:, :, 1],
    ),
    axis=1,
)
subspecies_bkg_pval_distribution = compute_class_pval(
    predictions=predictions,
    class_bkg_ts_distributions=class_bkg_ts_distributions,
    feature_bkg_distributions=feature_bkg_distributions,
    feature_sig_distributions=feature_sig_distributions,
    subspecies_feature_definitions=subspecies_feature_definitions,
)

reduce_func = lambda x: np.quantile(x, 0.8, axis=1)
reduce_func = lambda x: np.sum(x, axis=1)
# reduce_func = lambda x: np.sum(x < 0.1, axis=1)

p_values = compute_pval(
    predictions=predictions,
    subspecies_bkg_pval_distribution=subspecies_bkg_pval_distribution,
    class_bkg_ts_distributions=class_bkg_ts_distributions,
    feature_bkg_distributions=feature_bkg_distributions,
    feature_sig_distributions=feature_sig_distributions,
    subspecies_feature_definitions=subspecies_feature_definitions,
    reduce_func=reduce_func,
)
p_values_hybrid = compute_pval(
    predictions_hybrid[:, :, 1],
    subspecies_bkg_pval_distribution,
    class_bkg_ts_distributions,
    feature_bkg_distributions,
    feature_sig_distributions,
    subspecies_feature_definitions,
    reduce_func=reduce_func,
)

subspecies_bkg_pval_distribution.shape, p_values.shape, p_values_hybrid.shape
p_values

In [ ]:
feature_bkg_distributions[0]

In [ ]:
predictions_hybrid[:, :, 1].mean(axis=0)  # 1,

In [ ]:
p_bkg = [
    0.95864486,
    0.9625,
    0.9625,
    1.0,
    1.0,
    1.0,
    1,
    1,
]
np.searchsorted(p_bkg, [0.1, 0.96, 0.95864486, 1.0], side="right") / len(p_bkg)

In [ ]:
np.min(p_values), np.max(p_values)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
bins = np.linspace(0, 1, 100)
ax.hist(p_values, bins=bins, histtype="step", label="Non-Hybrid")
ax.hist(p_values_hybrid, bins=bins, histtype="step", label="Hybrid", ls="--")
ax.set_yscale("log")
ax.legend()

In [ ]:
subspecies_bkg_pval_distribution.shape

In [61]:
%matplotlib inline

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 10))
axes_flat = axes.flatten()

bins = np.linspace(0, 10, 50)

for key, values in class_ts_distributions.items():
    ax = axes_flat[key]

    # for feature in range(16):
    #     if not key in subspecies_feature_definitions[feature]:
    #         continue
    #     ax.hist(
    #         values[:, key, feature],
    #         bins=bins,
    #         histtype="step",
    #         label=f"Subspecies {key} | Feature {feature}",
    #     )
    # ax.hist(
    #     np.sum(values[:, key], axis=1),
    #     bins=bins,
    #     alpha=0.5,
    #     linewidth=2,
    #     label=f"Subspecies {key} [N={len(values)}]",
    # )
    ax.hist(
        class_bkg_ts_distributions[key],
        bins=bins,
        alpha=0.5,
        linewidth=2,
        label=f"Subspecies {key} [N={len(values)}]",
    )
    for subspecies in range(14):
        if subspecies == key:
            continue
        ax.hist(
            np.sum(values[:, subspecies], axis=1),
            bins=bins,
            histtype="step",
            ls="--",
        )
    ax.set_title("Class Background TS Distributions")
    ax.set_yscale("log")
    ax.set_xlabel("TS Value")
    ax.set_ylabel("Count")
    ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "class_background_ts_distributions.png"))